# Gate 1 - Decision Layer, SESOI, Influence Governance

Mục tiêu notebook:

- Đọc đúng Gate 0 source of truth đã khóa.
- Chạy Gate 1 workflow chính thức từ code trong repo bằng `python -m src.cli gate1`.
- Sinh các artifact: `gate1_inputs.json`, `gate1_input_integrity.json`, `n_eff_statement.json`, `simulation_registry.json`, `sesoi_registry.json`, `influence_rule.json`, `decision_memo.md`, `gate1_summary.json`.
- Kiểm tra rõ rằng Gate 1 không chạy model thật và không authorize real-data substantive phases.

Giới hạn liêm chính khoa học:

- Gate 1 là decision-layer simulation/governance, không phải real EEG model result.
- `N_primary_eligible = 15` là technical eligibility từ Gate 0, không phải bằng chứng hiệu quả mô hình.
- 68 sessions là repeated measurements, không phải primary independent denominator.
- Phase 0.5/1/2/3 real-data vẫn bị chặn cho đến sau Gate 2 và Gate 2.5.


In [ ]:
# ============================================================
# Gate 1 - Cell 1: Mount Drive, khai báo đường dẫn chuẩn
# ============================================================
# Mục tiêu:
# - Mount Google Drive.
# - Khai báo repo local trên Colab.
# - Khai báo Gate 0 source of truth đã signal-audit-ready.
# - Khai báo output root cho Gate 1.
# ============================================================

from pathlib import Path
from google.colab import drive

# Mount Drive. Nếu đã mount, Colab sẽ báo Drive already mounted.
drive.mount('/content/drive')

REPO_DIR = Path('/content/eeg-ds004752')
GATE0_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate0/20260417T102811097110Z')
GATE1_ROOT = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate1')

print('Repo:', REPO_DIR)
print('Gate 0 source of truth:', GATE0_RUN)
print('Gate 1 output root:', GATE1_ROOT)

for path in [REPO_DIR, GATE0_RUN]:
    print(path, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing required path: {path}')


In [ ]:
# ============================================================
# Gate 1 - Cell 2: Pull code mới nhất và chạy unit tests
# ============================================================
# Mục tiêu:
# - Lấy code mới nhất từ GitHub private repo.
# - Chạy unit tests trước khi tạo artifact Gate 1.
#
# Lưu ý:
# - Nếu repo private cần token, nhập GitHub token khi được hỏi.
# - Không in token ra output.
# ============================================================

import os
import base64
import getpass
import subprocess

%cd /content/eeg-ds004752

def run(cmd, cwd=REPO_DIR, check=True):
    print('\n$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)

pull_result = subprocess.run(['git', 'pull'], cwd=REPO_DIR)
if pull_result.returncode != 0:
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('git pull cần GitHub token. Nhập token: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    extra_header = f'Authorization: Basic {basic}'
    run(['git', '-c', f'http.extraHeader={extra_header}', 'pull'])

# In commit để truy vết. Cần có commit 54db0a0 hoặc mới hơn.
run(['git', 'log', '--oneline', '-5'])

# Chạy toàn bộ unit tests. Sau Gate 2, kỳ vọng hiện tại tối thiểu là 21 tests OK.
run(['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# ============================================================
# Gate 1 - Cell 3: Preflight kiểm tra Gate 0 source of truth
# ============================================================
# Mục tiêu:
# - Đảm bảo Gate 0 đã signal_audit_ready.
# - Đảm bảo không phải sample/batch audit.
# - Đảm bảo EDF/MAT đã materialize đủ.
# - Đảm bảo cohort lock sẵn sàng.
# ============================================================

import json

MANIFEST_PATH = GATE0_RUN / 'manifest.json'
COHORT_LOCK_PATH = GATE0_RUN / 'cohort_lock.json'
MATERIALIZATION_PATH = GATE0_RUN / 'materialization_report.json'
AUDIT_REPORT_PATH = GATE0_RUN / 'audit_report.md'

required_gate0_files = [
    MANIFEST_PATH,
    COHORT_LOCK_PATH,
    MATERIALIZATION_PATH,
    AUDIT_REPORT_PATH,
]

print('================ Gate 0 Artifact Check ================')
for path in required_gate0_files:
    print(path.name, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing Gate 0 artifact: {path}')

manifest = json.loads(MANIFEST_PATH.read_text())
cohort = json.loads(COHORT_LOCK_PATH.read_text())
signal = manifest.get('signal_audit', {})
payload = manifest.get('payload_state', {})

errors = []
if manifest.get('manifest_status') != 'signal_audit_ready':
    errors.append(f"manifest_status invalid: {manifest.get('manifest_status')}")
if signal.get('status') != 'ok':
    errors.append(f"signal_audit.status invalid: {signal.get('status')}")
if signal.get('subject_filter'):
    errors.append('Gate 0 signal audit has subject_filter; full audit required')
if signal.get('session_filter'):
    errors.append('Gate 0 signal audit has session_filter; full audit required')
if signal.get('candidate_sessions') != manifest.get('subjects', {}).get('n_sessions'):
    errors.append('candidate_sessions does not match manifest subjects.n_sessions')
if signal.get('sessions_checked') != manifest.get('subjects', {}).get('n_sessions'):
    errors.append('sessions_checked does not match manifest subjects.n_sessions')
if signal.get('candidate_mat_files') != payload.get('mat', {}).get('count'):
    errors.append('candidate_mat_files does not match manifest MAT count')
if signal.get('mat_files_checked') != payload.get('mat', {}).get('count'):
    errors.append('mat_files_checked does not match manifest MAT count')
if manifest.get('gate0_blockers') != []:
    errors.append(f"Gate 0 blockers not empty: {manifest.get('gate0_blockers')}")
if payload.get('edf', {}).get('pointer_like_count') != 0:
    errors.append('EDF payloads are not fully materialized')
if payload.get('mat', {}).get('pointer_like_count') != 0:
    errors.append('MAT derivatives are not fully materialized')
if cohort.get('cohort_lock_status') != 'signal_audit_ready':
    errors.append(f"cohort_lock_status invalid: {cohort.get('cohort_lock_status')}")
if cohort.get('n_primary_eligible') != 15:
    errors.append(f"n_primary_eligible expected 15, got {cohort.get('n_primary_eligible')}")

if errors:
    print('Gate 1 preflight FAILED:')
    for item in errors:
        print('-', item)
    raise SystemExit('Stop Gate 1: Gate 0 source is not valid.')

print('\nGate 1 preflight PASSED.')
print('Manifest status:', manifest['manifest_status'])
print('Signal status:', signal['status'])
print('Sessions checked:', signal['sessions_checked'])
print('MAT files checked:', signal['mat_files_checked'])
print('Gate 0 blockers:', manifest['gate0_blockers'])
print('Cohort lock status:', cohort['cohort_lock_status'])
print('N primary eligible:', cohort['n_primary_eligible'])


In [ ]:
# ============================================================
# Gate 1 - Cell 4: Chạy Gate 1 chính thức bằng CLI
# ============================================================
# Mục tiêu:
# - Chạy workflow versioned trong repo, không tự tạo logic trong notebook.
# - Sinh đầy đủ Gate 1 artifacts vào Google Drive.
#
# Artifact được sinh:
# - gate1_inputs.json
# - gate1_input_integrity.json
# - n_eff_statement.json
# - simulation_registry.json
# - sesoi_registry.json
# - influence_rule.json
# - decision_memo.md
# - gate1_summary.json
# ============================================================

run([
    'python', '-m', 'src.cli', 'gate1',
    '--profile', 't4_safe',
    '--gate0-run', str(GATE0_RUN),
    '--config', 'configs/gate1/decision_simulation.json',
    '--output-root', str(GATE1_ROOT),
])


In [ ]:
# ============================================================
# Gate 1 - Cell 5: Validate gate1_summary.json
# ============================================================
# Mục tiêu:
# - Đọc latest Gate 1 run.
# - Xác nhận decision layer ready.
# - Xác nhận không authorize real-data phase.
# ============================================================

LATEST_GATE1 = GATE1_ROOT / 'latest.txt'
if not LATEST_GATE1.exists():
    raise FileNotFoundError(f'Missing Gate 1 latest pointer: {LATEST_GATE1}')

gate1_run = Path(LATEST_GATE1.read_text().strip())
summary_path = gate1_run / 'gate1_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing gate1_summary.json: {summary_path}')

summary = json.loads(summary_path.read_text())

print('================ Gate 1 Official Summary ================')
print('Run dir:', gate1_run)
print('Status:', summary['status'])
print('Gate 0 source:', summary['gate0_source_of_truth'])
print('Git commit:', summary['git_commit'])
print('Working tree clean:', summary['working_tree_clean'])

print('\nN_eff:')
print('N primary eligible:', summary['n_eff']['n_primary_eligible'])
print('Outer folds planned:', summary['n_eff']['n_outer_folds_planned'])
print('Primary denominator:', summary['n_eff']['primary_denominator'])

print('\nSimulation:')
print('Status:', summary['simulation']['status'])
print('Scenario count:', summary['simulation']['scenario_count'])
print('Repeats:', summary['simulation']['n_repeats'])
print('Not model result:', summary['simulation']['not_a_model_result'])

print('\nSESOI:')
print('Subject-level Delta BA:', summary['sesoi']['subject_level_sesoi_delta_ba'])
print('Max allowed Delta ECE:', summary['sesoi']['max_allowed_delta_ece'])

print('\nInfluence:')
print('Influence ceiling:', summary['influence']['influence_ceiling'])
print('Unit:', summary['influence']['unit'])

print('\nGovernance:')
print('Real-data phase authorized:', summary['real_data_phase_authorized'])
print('Next gate:', summary['next_gate'])

assert summary['status'] == 'gate1_decision_layer_ready'
assert summary['n_eff']['n_primary_eligible'] == 15
assert summary['n_eff']['primary_denominator'] == 'subject'
assert summary['simulation']['scenario_count'] == 27
assert summary['simulation']['n_repeats'] == 5000
assert summary['simulation']['not_a_model_result'] is True
assert summary['sesoi']['subject_level_sesoi_delta_ba'] == 0.03
assert summary['sesoi']['max_allowed_delta_ece'] == 0.02
assert summary['influence']['influence_ceiling'] == 0.40
assert summary['real_data_phase_authorized'] is False
assert summary['next_gate'] == 'gate2_synthetic_validation'

print('\nGate 1 official summary validation PASSED.')


In [ ]:
# ============================================================
# Gate 1 - Cell 6: Kiểm tra artifact và preview decision memo
# ============================================================
# Mục tiêu:
# - Xác nhận các artifact bắt buộc có mặt.
# - Preview decision memo để kiểm tra language governance.
# ============================================================

required_gate1_files = [
    'gate1_inputs.json',
    'gate1_input_integrity.json',
    'n_eff_statement.json',
    'simulation_registry.json',
    'sesoi_registry.json',
    'influence_rule.json',
    'decision_memo.md',
    'gate1_summary.json',
]

print('================ Gate 1 Artifacts ================')
for name in required_gate1_files:
    path = gate1_run / name
    print(name, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing Gate 1 artifact: {path}')

integrity = json.loads((gate1_run / 'gate1_input_integrity.json').read_text())
simulation_registry = json.loads((gate1_run / 'simulation_registry.json').read_text())
sesoi_registry = json.loads((gate1_run / 'sesoi_registry.json').read_text())
influence_rule = json.loads((gate1_run / 'influence_rule.json').read_text())

print('\n================ Integrity ================')
print('Status:', integrity['status'])
print('Git commit:', integrity['repo']['commit'])
print('Working tree clean:', integrity['repo']['working_tree_clean'])
print('Manifest hash:', integrity['artifact_hashes_sha256']['manifest_json']['sha256'])
print('Cohort hash:', integrity['artifact_hashes_sha256']['cohort_lock_json']['sha256'])

print('\n================ Simulation Registry ================')
print('Status:', simulation_registry['status'])
print('Simulation type:', simulation_registry['simulation_type'])
print('Scenario count:', simulation_registry['scenario_count'])
print('Interpretation:', simulation_registry['interpretation'])
print('First scenario:', json.dumps(simulation_registry['scenarios'][0], indent=2))

print('\n================ SESOI Registry ================')
print(json.dumps(sesoi_registry, indent=2))

print('\n================ Influence Rule ================')
print(json.dumps(influence_rule, indent=2))

print('\n================ Decision Memo Preview ================')
decision_memo = (gate1_run / 'decision_memo.md').read_text()
print(decision_memo[:3500])


In [ ]:
# ============================================================
# Gate 1 - Cell 7: Kết luận đúng phạm vi
# ============================================================
# Mục tiêu:
# - In kết luận không overclaim.
# - Chỉ xác nhận đủ điều kiện kỹ thuật sang Gate 2 synthetic validation.
# ============================================================

print('================ Interpretation ================')
print('OK: Gate 1 decision layer is ready.')
print('OK: N_eff, SESOI, decision assumptions, influence governance, and pivot policy are locked.')
print('OK: Project can proceed to Gate 2 synthetic validation.')
print('NOT OK TO CLAIM: Gate 1 does not prove real EEG model efficacy or privileged transfer.')
print('BLOCKED: Phase 0.5/1/2/3 real-data substantive runs remain blocked until Gate 2 and Gate 2.5 pass.')
print('\nGate 1 source of truth:', gate1_run)
